# TPR Risk Management Automation
## VFS IT - Technology Performance Review

**Purpose**: Automate consolidation of risk data from 5 business areas and generate charts

---

### Instructions:
1. **Place your Excel files** in `data/input/risk/` folder
2. **Run all cells** (Cell → Run All)
3. **Check outputs** in `data/output/` folder

---

## Step 1: Import Required Libraries

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

print("✅ Libraries imported successfully!")

## Step 2: Configuration

In [ ]:
# Configuration
INPUT_FOLDER = Path('data/input/risk')
OUTPUT_FOLDER = Path('data/output')

# Create folders if they don't exist
INPUT_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Business areas (update with your actual tech lead names)
BUSINESS_AREAS = [
    "Tech Lead 1",
    "Tech Lead 2",
    "Tech Lead 3",
    "Tech Lead 4",
    "Tech Lead 5"
]

print("✅ Configuration set!")
print(f"📁 Input folder: {INPUT_FOLDER.absolute()}")
print(f"📁 Output folder: {OUTPUT_FOLDER.absolute()}")

## Step 3: Find and Load Excel Files

In [ ]:
# Find all Excel files
excel_files = list(INPUT_FOLDER.glob('*.xlsx')) + list(INPUT_FOLDER.glob('*.xls'))
risk_files = [f for f in excel_files if 'Operational Risk Assessment' in f.name]

print(f"\n📂 Found {len(risk_files)} risk assessment files:")
for file in risk_files:
    print(f"   ✓ {file.name}")

if not risk_files:
    print("\n⚠️  No Excel files found!")
    print(f"Please place your Excel files in: {INPUT_FOLDER.absolute()}")
    print("Expected naming: 'Operational Risk Assessment - [Name].xlsx'")

## Step 4: Consolidate Data

In [ ]:
# Read and consolidate all files
all_data = []

for file_path in risk_files:
    try:
        # Read Excel file
        df = pd.read_excel(file_path, sheet_name=0)
        
        # Extract business area from filename
        business_area = file_path.name.replace('Operational Risk Assessment - ', '').replace('.xlsx', '').replace('.xls', '').strip()
        
        # Add business area column
        df.insert(0, 'Business Area', business_area)
        
        all_data.append(df)
        print(f"✓ Loaded {len(df)} risks from {business_area}")
        
    except Exception as e:
        print(f"❌ Error reading {file_path.name}: {str(e)}")

if all_data:
    # Combine all dataframes
    consolidated_df = pd.concat(all_data, ignore_index=True)
    
    # Clean data
    if 'Status' in consolidated_df.columns:
        consolidated_df['Status'] = consolidated_df['Status'].str.strip().str.title()
        consolidated_df['Status'] = consolidated_df['Status'].replace({'Opened': 'Open'})
    
    if 'Risk Rating' in consolidated_df.columns:
        consolidated_df['Risk Rating'] = consolidated_df['Risk Rating'].str.strip().str.title()
    
    if 'Category' in consolidated_df.columns:
        consolidated_df['Category'] = consolidated_df['Category'].str.strip()
    
    print(f"\n✅ Consolidation complete: {len(consolidated_df)} total risks from {len(risk_files)} business areas")
    
    # Display sample
    print("\n📊 Sample of consolidated data:")
    display(consolidated_df.head())
else:
    print("❌ No data could be loaded!")

## Step 5: Save Master Spreadsheet

In [ ]:
# Save to Excel
master_file = OUTPUT_FOLDER / 'Master_Risk_Tracker.xlsx'

with pd.ExcelWriter(master_file, engine='openpyxl') as writer:
    consolidated_df.to_excel(writer, sheet_name='Risk Data', index=False)

print(f"✅ Master spreadsheet saved: {master_file}")
print(f"   Total risks: {len(consolidated_df)}")

## Step 6: Generate Statistics

In [ ]:
# Calculate statistics
stats = {
    'Total Risks': len(consolidated_df),
    'Open Risks': len(consolidated_df[consolidated_df['Status'] == 'Open']),
    'Closed Risks': len(consolidated_df[consolidated_df['Status'] == 'Closed']),
    'Business Areas': consolidated_df['Business Area'].nunique()
}

print("\n📊 Summary Statistics:")
print("=" * 50)
for key, value in stats.items():
    print(f"{key:20}: {value}")

print("\n📈 Risks by Priority:")
if 'Risk Rating' in consolidated_df.columns:
    print(consolidated_df['Risk Rating'].value_counts())

print("\n📈 Risks by Category:")
if 'Category' in consolidated_df.columns:
    print(consolidated_df['Category'].value_counts())

## Step 7: Generate Chart 1 - Stacked Bar (Open Risks by Priority)

In [ ]:
# Filter for open risks only
open_risks = consolidated_df[consolidated_df['Status'] == 'Open'].copy()

if not open_risks.empty and 'Risk Rating' in open_risks.columns:
    # Create pivot table
    pivot_data = pd.crosstab(open_risks['Business Area'], open_risks['Risk Rating'])
    
    # Ensure priority order
    priority_order = ['High', 'Medium', 'Low']
    existing_priorities = [p for p in priority_order if p in pivot_data.columns]
    pivot_data = pivot_data[existing_priorities]
    
    # Create chart
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = {'High': '#d32f2f', 'Medium': '#ff9800', 'Low': '#4caf50'}
    colors_list = [colors.get(p, '#808080') for p in existing_priorities]
    
    pivot_data.plot(kind='bar', stacked=True, ax=ax, color=colors_list, width=0.7)
    
    ax.set_title('Open Risks Priorities per Business Area', fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Business Area', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Open Risks', fontsize=12, fontweight='bold')
    ax.legend(title='Priority', loc='upper right')
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels
    for container in ax.containers:
        ax.bar_label(container, label_type='center', fontsize=9)
    
    plt.tight_layout()
    
    # Save
    chart1_path = OUTPUT_FOLDER / 'risk_priorities_stacked_bar.png'
    plt.savefig(chart1_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Chart 1 saved: {chart1_path}")
else:
    print("⚠️  No open risks found for stacked bar chart")

## Step 8: Generate Chart 2 - Pie (Risk Statuses)

In [ ]:
if 'Status' in consolidated_df.columns:
    status_counts = consolidated_df['Status'].value_counts()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    colors = {'Open': '#ff6b6b', 'Closed': '#51cf66'}
    pie_colors = [colors.get(status, '#808080') for status in status_counts.index]
    
    wedges, texts, autotexts = ax.pie(
        status_counts.values,
        labels=status_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=pie_colors,
        explode=[0.05] * len(status_counts)
    )
    
    ax.set_title('Operational Risk Statuses', fontsize=14, fontweight='bold', pad=20)
    
    for text in texts:
        text.set_fontsize(11)
        text.set_fontweight('bold')
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')
    
    legend_labels = [f"{status}: {count}" for status, count in status_counts.items()]
    ax.legend(legend_labels, loc='upper left', bbox_to_anchor=(1, 0, 0.5, 1))
    
    plt.tight_layout()
    
    chart2_path = OUTPUT_FOLDER / 'risk_statuses_pie.png'
    plt.savefig(chart2_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Chart 2 saved: {chart2_path}")
else:
    print("⚠️  Status column not found")

## Step 9: Generate Chart 3 - Pie (Open Risk Categories)

In [ ]:
if not open_risks.empty and 'Category' in open_risks.columns:
    category_counts = open_risks['Category'].value_counts()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    category_colors = {
        'Resources': '#e91e63',
        'Cyber': '#9c27b0',
        'Stability': '#3f51b5',
        'Delivery': '#00bcd4',
        'Budget': '#4caf50',
        '3rd Party': '#ff9800'
    }
    pie_colors = [category_colors.get(cat, '#808080') for cat in category_counts.index]
    
    wedges, texts, autotexts = ax.pie(
        category_counts.values,
        labels=category_counts.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=pie_colors,
        explode=[0.03] * len(category_counts)
    )
    
    ax.set_title('% Split Open Risk Categories', fontsize=14, fontweight='bold', pad=20)
    
    for text in texts:
        text.set_fontsize(10)
        text.set_fontweight('bold')
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontsize(9)
        autotext.set_fontweight('bold')
    
    legend_labels = [f"{cat}: {count}" for cat, count in category_counts.items()]
    ax.legend(legend_labels, loc='upper left', bbox_to_anchor=(1, 0, 0.5, 1))
    
    plt.tight_layout()
    
    chart3_path = OUTPUT_FOLDER / 'risk_categories_pie.png'
    plt.savefig(chart3_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Chart 3 saved: {chart3_path}")
else:
    print("⚠️  No open risks found for categories pie chart")

## Step 10: Summary

In [ ]:
print("\n" + "="*70)
print(" " * 20 + "✅ AUTOMATION COMPLETE!")
print("="*70)

print(f"\n📊 Files Generated in {OUTPUT_FOLDER.absolute()}:")
print("   1. Master_Risk_Tracker.xlsx")
print("   2. risk_priorities_stacked_bar.png")
print("   3. risk_statuses_pie.png")
print("   4. risk_categories_pie.png")

print(f"\n📈 Summary:")
print(f"   Total Risks: {stats['Total Risks']}")
print(f"   Open: {stats['Open Risks']}")
print(f"   Closed: {stats['Closed Risks']}")
print(f"   Business Areas: {stats['Business Areas']}")

print("\n💡 Next Steps:")
print("   - Open the Excel file to review consolidated data")
print("   - Copy the charts to your TPR PowerPoint presentation")
print("   - Save time and celebrate! 🎉")

print("\n" + "="*70)